[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap04_inflacao.ipynb)

# Capítulo 4: O dragão da inflação

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Neste laboratório você baixa o IPCA direto do Banco Central, vê quanto o dinheiro perdeu de poder de compra desde o Plano Real e calcula o juro real: o rendimento que sobra depois da inflação.

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. Baixar o IPCA mensal
A série 433 do Banco Central traz a variação mensal do IPCA, calculado pelo IBGE.

In [ ]:
ipca = sgs(433, inicio='01/07/1994') / 100
print('Primeiro mês:', ipca.index[0].strftime('%m/%Y'), '| Último mês:', ipca.index[-1].strftime('%m/%Y'))
ipca.tail(12).map(pct)

## 2. Quanto R$ 100 perderam desde uma data
Escolha a data de início. Experimente o mês do seu primeiro salário.

In [ ]:
INICIO = '1994-07'   # MUDE AQUI (ano-mês)
VALOR = 100          # MUDE AQUI

serie = ipca[INICIO:]
fator = (1 + serie).prod()
print(f'Inflação acumulada desde {INICIO}: {pct(fator - 1)}')
print(f'{brl(VALOR)} de {INICIO} compram hoje o mesmo que {brl(VALOR / fator)} compravam na época.')
print(f'Para comprar o mesmo que {brl(VALOR)} compravam, hoje você precisa de {brl(VALOR * fator)}.')

poder = VALOR / (1 + serie).cumprod()
ax = poder.plot(color=CORES['vermelho'], lw=2, title=f'Poder de compra de {brl(VALOR,0)} guardados no colchão desde {INICIO}')
eixo_reais(ax); ax.set_xlabel(''); plt.show()

## 3. A inflação de 12 meses
O Banco Central persegue uma meta de 3% ao ano, com tolerância de 1,5 ponto para cima ou para baixo (meta contínua desde 2025). Veja como a inflação se comportou.

In [ ]:
ipca12 = (1 + ipca).rolling(12).apply(np.prod, raw=True) - 1
ax = (ipca12['2005':] * 100).plot(color=CORES['azul'], lw=2, title='IPCA acumulado em 12 meses (%)')
ax.axhline(3, color=CORES['verde'], ls='--', label='Meta de 3%')
ax.axhspan(1.5, 4.5, color=CORES['verde'], alpha=0.08, label='Intervalo de tolerância (2025 em diante)')
ax.legend(); ax.set_xlabel(''); plt.show()
print('Último IPCA de 12 meses:', pct(ipca12.iloc[-1]))

## 4. O único juro que importa: o juro real
O CDI é a referência da renda fixa. A série 4391 traz o CDI acumulado de cada mês. O juro real é (1 + rendimento) ÷ (1 + inflação) − 1.

In [ ]:
cdi = sgs(4391, inicio='01/01/2000') / 100
df = pd.concat({'cdi': cdi, 'ipca': ipca}, axis=1).dropna()
cdi12 = (1 + df['cdi']).rolling(12).apply(np.prod, raw=True) - 1
ipc12 = (1 + df['ipca']).rolling(12).apply(np.prod, raw=True) - 1
real12 = (1 + cdi12) / (1 + ipc12) - 1
ax = (real12['2005':] * 100).plot(color=CORES['azul'], lw=2, title='Juro real do CDI em 12 meses, antes do imposto (%)')
ax.axhline(0, color=CORES['vermelho'], lw=1)
ax.set_xlabel(''); plt.show()
print('Juro real do CDI nos últimos 12 meses, antes do imposto:', pct(real12.iloc[-1]))

**Para pensar:** quando a linha fica abaixo de zero, quem deixou o dinheiro no CDI perdeu poder de compra, mesmo vendo o saldo subir. Faça a mesma conta com a sua aplicação na [calculadora de inflação e poder de compra](https://www.traderbrasil.com/blog/calculadora-inflacao-ipca-poder-de-compra.php).

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 4 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap04/

Achou um erro? Escreva para contato@traderbrasil.com.